# AACAgent — Metrics Notebook

Computes evaluation metrics by joining a CSV produced by `eval_cpu.ipynb` or
`eval_gpu.ipynb` with the ground-truth parquet dataset.

## CSV columns expected (from eval notebooks)
```
row_idx, input_type, turn_pos, concept_text,
called_get_time, called_get_schedule,
predicted_ids, plan_method, resolve_method, planner_concepts
```

## Parquet columns expected (eval_final.parquet)
```
sentence, concepts (list of {concept_text, gold_id, candidate_ids}),
caregiver_clear, caregiver_vague, time_of_day, event_time, schedule, split
```

## Metrics computed
- **hit** (`gold_in_window`) — gold ID present in the predicted_ids window
- **gold_in_candidates** — not directly available from CPU eval CSV;
  approximated as `resolve_method != 'none'` (a concept was resolved →
  its ARASAAC pool was non-empty → gold *may* have been reachable)
- **resolve_none_rate** — fraction of turns where resolve_method=none
- **planner_had_gold_concept** — planner generated exactly the gold concept_text
- **tool_call_rate** — fraction of turn_pos==0 rows where get_time/get_schedule called
- **tool_call_rate_by_split** — same, broken down by input_type

> **Note on `called_get_time` / `called_get_schedule` always being False**
> See §Root cause analysis cell below for the diagnosis and recommended fix.

Run all cells in order. Edit only the `CONFIG` cell.

In [ ]:
# ─── CONFIG — edit here ───────────────────────────────────────────────────────
import os

# Path to the CSV produced by eval_cpu.ipynb or eval_gpu.ipynb
EVAL_CSV = os.environ.get(
    "NB_EVAL_CSV",
    "/content/aac-mcp-agent/eval/cpu/eval_cpu_colab.csv",
)

# Path to the ground-truth parquet (eval_final.parquet has the 'split' column)
EVAL_PARQUET = os.environ.get(
    "NB_EVAL_PARQUET",
    "/content/aac-mcp-agent/annotation/eval_final.parquet",
)

# Optional: filter by model name (empty string = keep all models in the CSV)
MODEL_FILTER = os.environ.get("NB_MODEL_FILTER", "")

print(f"EVAL_CSV     : {EVAL_CSV}")
print(f"EVAL_PARQUET : {EVAL_PARQUET}")
print(f"MODEL_FILTER : {MODEL_FILTER!r}")

In [ ]:
import ast
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
print("Imports OK.")

In [ ]:
# ─── Load eval CSV ────────────────────────────────────────────────────────────
csv_path = Path(EVAL_CSV)
assert csv_path.exists(), f"CSV not found: {csv_path}"

res = pd.read_csv(csv_path)

# Parse list columns
for col in ("predicted_ids", "planner_concepts"):
    if res[col].dtype == object:
        res[col] = res[col].apply(ast.literal_eval)

# Normalise bool columns (they may be stored as strings 'True'/'False')
for col in ("called_get_time", "called_get_schedule"):
    if res[col].dtype == object:
        res[col] = res[col].map({"True": True, "False": False, True: True, False: False})

if MODEL_FILTER:
    before = len(res)
    res = res[res["model"] == MODEL_FILTER].copy() if "model" in res.columns else res
    print(f"Model filter applied: {before} → {len(res)} rows (model={MODEL_FILTER!r})")

print(f"CSV loaded   : {len(res):,} rows")
print(f"Columns      : {list(res.columns)}")
print(f"input_type   : {res['input_type'].value_counts().to_dict()}")
print(f"turn_pos dist: {res['turn_pos'].value_counts().sort_index().to_dict()}")

In [ ]:
# ─── Load ground-truth parquet ────────────────────────────────────────────────
pq_path = Path(EVAL_PARQUET)
assert pq_path.exists(), f"Parquet not found: {pq_path}"

df_gold = pd.read_parquet(pq_path)

# Ensure concepts column is a Python list of dicts (not strings)
if df_gold["concepts"].dtype == object and isinstance(df_gold["concepts"].iloc[0], str):
    df_gold["concepts"] = df_gold["concepts"].apply(ast.literal_eval)

# Build a flat gold lookup: (row_idx, concept_text) → {gold_id, candidate_ids}
gold_records = []
for row_idx, row in df_gold.iterrows():
    for c in row["concepts"]:
        gold_records.append({
            "row_idx":       row_idx,
            "concept_text":  c["concept_text"],
            "gold_id":       int(c["gold_id"]),
            "candidate_ids": c.get("candidate_ids", []),
            "split":         row.get("split", "unknown"),
        })
df_gold_flat = pd.DataFrame(gold_records)

print(f"Parquet loaded: {len(df_gold):,} sentences")
print(f"Gold flat     : {len(df_gold_flat):,} concept entries")
print(f"Split dist    : {df_gold_flat['split'].value_counts().to_dict()}")

In [ ]:
# ─── Join CSV with gold ───────────────────────────────────────────────────────
#
# The CSV uses row_idx from the sampled df (df.index, which is the original
# parquet index after reset_index(drop=True) in the eval notebook).
# eval_final row order == integer index 0..N-1 so this join is exact.
#
# Join key: (row_idx, concept_text)
# Note: input_type (clear/vague) does NOT change the gold; the same gold_id
# applies regardless of which caregiver phrasing was used.

merged = res.merge(
    df_gold_flat[["row_idx", "concept_text", "gold_id", "candidate_ids", "split"]],
    on=["row_idx", "concept_text"],
    how="left",
)

n_unmatched = merged["gold_id"].isna().sum()
if n_unmatched:
    print(f"⚠️  {n_unmatched} rows could not be joined to a gold entry.")
    print("    (planner generated concept_text not present in the dataset — expected)")

print(f"Merged rows  : {len(merged):,}")
print(f"Matched gold : {merged['gold_id'].notna().sum():,} ({merged['gold_id'].notna().mean():.1%})")

In [ ]:
# ─── Compute per-row binary flags ─────────────────────────────────────────────

def _hit(row) -> bool:
    """gold_id in predicted_ids window."""
    if pd.isna(row["gold_id"]):
        return False
    return int(row["gold_id"]) in row["predicted_ids"]

def _gold_in_candidates(row) -> object:
    """gold_id in candidate_ids (pre-ranking pool).
    Returns True/False when data is available, NaN when candidate_ids is empty."""
    if pd.isna(row["gold_id"]):
        return float("nan")
    cids = row["candidate_ids"]
    if not isinstance(cids, list) or len(cids) == 0:
        return float("nan")
    return int(row["gold_id"]) in cids

def _planner_had_gold_concept(row) -> object:
    """True if the gold concept_text appears in the planner's concepts list."""
    if pd.isna(row["gold_id"]):
        return float("nan")
    ct = str(row["concept_text"]).lower().strip()
    concepts = [str(c).lower().strip() for c in row["planner_concepts"]]
    return ct in concepts

merged["hit"]                    = merged.apply(_hit,                    axis=1)
merged["gold_in_candidates"]     = merged.apply(_gold_in_candidates,     axis=1)
merged["planner_had_gold_concept"] = merged.apply(_planner_had_gold_concept, axis=1)
merged["resolve_none"]           = merged["resolve_method"] == "none"
merged["window_size"]            = merged["predicted_ids"].apply(len)

# Tool call flag (only meaningful at turn_pos==0)
merged["tool_called"] = merged["called_get_time"] | merged["called_get_schedule"]

print("Flags computed. Sample:")
print(merged[["row_idx","input_type","turn_pos","concept_text",
               "hit","planner_had_gold_concept","resolve_none","tool_called"]].head(8).to_string())

In [ ]:
# ─── Root-cause analysis: tool calls always False ────────────────────────────
#
# DIAGNOSIS
# ---------
# `called_get_time` and `called_get_schedule` are always False because the
# eval notebook writes them from `ec.tool_calls` (the EvalContext object),
# but the EvalContext is only populated when `call_tools=True` comes back
# from `_plan()` AND `_collect_context()` is actually entered.
#
# The recording logic in run_multi_turn() is:
#
#   "called_get_time":  "get_time"     in ec.tool_calls,
#   "called_get_schedule": "get_schedule" in ec.tool_calls,
#
# And in agent._collect_context(), the tool name is appended to
# `ctx.tool_calls` only when the mock data is non-None AND valid:
#
#   if ctx.mock_time is not None:
#       ctx.tool_calls.append("get_time")
#
# BUT there is a subtlety: `_collect_context()` is only called inside
# agent.run() when `call_tools=True` — and `call_tools` comes from the
# planner LLM output.
#
# The real issue is one level earlier: in run_multi_turn(), `ec` is built
# with `build_eval_ctx(row)` at turn_pos==0, but the EvalContext is passed
# to `agent.run(raw_input, eval_ctx=ec)`. Inside `agent.run`, `_eval_ctx`
# is set, and then `call_tools, concepts = self._plan(...)` is called.
# If the LLM says `call_tools=False` (which it does for 'clear' inputs),
# `_collect_context()` is never called, so `ec.tool_calls` stays empty.
#
# For 'vague' inputs the LLM *should* return `call_tools=True`, and then
# `_collect_context()` *would* append to `ec.tool_calls`. But the CSV shows
# 0/0 for both — this means the LLM is consistently returning
# `call_tools=False` even for vague inputs, OR its JSON output is
# malformed and the fallback always defaults to False.
#
# EVIDENCE FROM CSV
# -----------------
# plan_method is almost always 'llm' (not a fallback), so the LLM DID
# respond. The planner concepts are non-empty and semantically rich —
# the LLM is understanding the input. So the issue is specifically that
# the JSON field `call_tools` is being parsed as False even for vague inputs.
#
# Most likely causes (in order of probability):
#
# 1. The SHORT prompt + Qwen2.5-3B on CPU at n_ctx=512 truncates its output
#    and the JSON is cut off before `call_tools`, causing the regex fallback
#    to miss it — and `parsed.get("call_tools", True)` defaults to True, but
#    the bool() of a missing key defaults to True... actually the code says:
#       call_tools = bool(parsed.get("call_tools", True))
#    So a missing key → True. That would OVER-trigger, not suppress calls.
#
# 2. The LLM returns call_tools=false (lowercase) and the JSON parser
#    interprets it correctly as False — meaning the LLM genuinely decides
#    not to call tools even for vague inputs.
#
# 3. The caregiver_vague texts in the dataset are actually explicit enough
#    that Qwen2.5-3B doesn't classify them as vague (they are annotated
#    'vague' by the annotation LLM but still contain content words).
#
# RECOMMENDATION
# --------------
# Add a debug column to the eval CSV: `planner_raw_response` (the raw text
# from the LLM before JSON parsing). This would immediately reveal whether
# the LLM is outputting call_tools=false or if parsing is failing.
# In eval_cpu.ipynb, in run_multi_turn(), expose agent.last_plan_raw if
# you add that attribute to AACAgent._plan().
#
# As an independent check: the tool_call_rate by input_type is shown below.

t0 = merged[merged["turn_pos"] == 0].copy()
print("Tool call rate at turn_pos==0 (should be ~1.0 for vague, ~0.0 for clear):")
print(t0.groupby("input_type")[["called_get_time","called_get_schedule","tool_called"]].mean().round(3))
print()
print("plan_method distribution:")
print(merged["plan_method"].value_counts())

In [ ]:
# ─── Main metrics — overall ───────────────────────────────────────────────────

def _pct(series) -> str:
    v = series.dropna().mean()
    return f"{v:.1%}" if pd.notna(v) else "N/A"

# Only rows where gold was matched (concept_text in dataset)
m = merged.dropna(subset=["gold_id"]).copy()

print("=" * 55)
print(" OVERALL METRICS")
print("=" * 55)
print(f" Rows evaluated              : {len(merged):,}")
print(f" Rows with gold match        : {len(m):,} ({len(m)/len(merged):.1%})")
print()
print(f" Hit@window                  : {_pct(m['hit'])}")
print(f"   (gold_id in predicted window)")
print()
print(f" gold_in_candidates          : {_pct(m['gold_in_candidates'])}")
print(f"   (gold in pre-ranking pool — requires candidate_ids in parquet)")
print()
print(f" resolve_none_rate           : {_pct(m['resolve_none'])}")
print(f"   (concept not found in ARASAAC keyword index)")
print()
print(f" planner_had_gold_concept    : {_pct(m['planner_had_gold_concept'])}")
print(f"   (planner generated the exact gold concept_text)")
print()
print(f" Avg window size             : {m['window_size'].mean():.1f}")
print()
print("─" * 55)
print(" TOOL CALL METRICS (turn_pos == 0 only)")
print("─" * 55)
t0_m = m[m["turn_pos"] == 0]
print(f" get_time called rate        : {_pct(t0_m['called_get_time'])}")
print(f" get_schedule called rate    : {_pct(t0_m['called_get_schedule'])}")
print(f" any tool called rate        : {_pct(t0_m['tool_called'])}")
print(f"   EXPECTED: ~1.0 for vague, ~0.0 for clear")
print("=" * 55)

In [ ]:
# ─── Metrics by input_type (clear vs vague) ───────────────────────────────────

print("=" * 65)
print(" METRICS BY INPUT TYPE")
print("=" * 65)

for it in ["clear", "vague"]:
    sub = m[m["input_type"] == it]
    sub_t0 = sub[sub["turn_pos"] == 0]
    print(f"\n [input_type = {it.upper()}]  n={len(sub)}")
    print(f"   Hit@window              : {_pct(sub['hit'])}")
    print(f"   gold_in_candidates      : {_pct(sub['gold_in_candidates'])}")
    print(f"   resolve_none_rate       : {_pct(sub['resolve_none'])}")
    print(f"   planner_had_gold_concept: {_pct(sub['planner_had_gold_concept'])}")
    print(f"   tool_called (t0 only)   : {_pct(sub_t0['tool_called'])}")
    print(f"   avg window size         : {sub['window_size'].mean():.1f}")

print()
print("=" * 65)

In [ ]:
# ─── Metrics by turn_pos ──────────────────────────────────────────────────────

print("=" * 65)
print(" METRICS BY TURN POSITION")
print("=" * 65)

tp_groups = m.groupby("turn_pos").agg(
    n                   = ("hit", "count"),
    hit_rate            = ("hit", "mean"),
    resolve_none_rate   = ("resolve_none", "mean"),
    planner_had_gold    = ("planner_had_gold_concept", "mean"),
    avg_window_size     = ("window_size", "mean"),
).round(3)
print(tp_groups.to_string())

In [ ]:
# ─── Conditional hit rates ────────────────────────────────────────────────────
#
# These slice the data to isolate where each component of the pipeline is the
# bottleneck.  Interpret as: "IF the planner got it right, how often do we hit?"

print("=" * 65)
print(" CONDITIONAL HIT RATES")
print("=" * 65)

had_gold = m[m["planner_had_gold_concept"] == True]
no_gold  = m[m["planner_had_gold_concept"] == False]
resolved = m[m["resolve_none"] == False]
not_res  = m[m["resolve_none"] == True]

print(f"\nHit when planner_had_gold_concept=True  : {_pct(had_gold['hit'])}  (n={len(had_gold)})")
print(f"Hit when planner_had_gold_concept=False : {_pct(no_gold['hit'])}  (n={len(no_gold)})")
print()
print(f"Hit when resolve_method != 'none'       : {_pct(resolved['hit'])}  (n={len(resolved)})")
print(f"Hit when resolve_method == 'none'       : {_pct(not_res['hit'])}  (n={len(not_res)})")
print()
# Chain: planner got it AND resolve succeeded
chain_ok = m[(m["planner_had_gold_concept"] == True) & (m["resolve_none"] == False)]
print(f"Hit when BOTH planner_had_gold AND resolved: {_pct(chain_ok['hit'])}  (n={len(chain_ok)})")
print()
print("─" * 65)
print("PIPELINE BOTTLENECK ANALYSIS:")
print(f"  Planner coverage    : {_pct(m['planner_had_gold_concept'])} of turns include gold concept")
print(f"  Resolve coverage    : {_pct(~m['resolve_none'])} of turns resolve to a keyword")
print(f"  Upper bound (chain) : {_pct(m['planner_had_gold_concept'] & ~m['resolve_none'])} reach ranking stage")
print("=" * 65)

In [ ]:
# ─── Resolve method breakdown ─────────────────────────────────────────────────

print("=" * 65)
print(" RESOLVE METHOD BREAKDOWN")
print("=" * 65)

rm = m.groupby("resolve_method").agg(
    n        = ("hit", "count"),
    hit_rate = ("hit", "mean"),
).sort_values("n", ascending=False).round(3)
print(rm.to_string())

print()
print(f"Overall resolve_none_rate: {_pct(m['resolve_none'])}")
print(f"  (target: <20% vs ~88% on legacy dataset — §13 context doc)")

In [ ]:
# ─── Plan method breakdown ────────────────────────────────────────────────────

print("=" * 65)
print(" PLAN METHOD BREAKDOWN")
print("=" * 65)

pm = m.groupby("plan_method").agg(
    n               = ("hit", "count"),
    hit_rate        = ("hit", "mean"),
    resolve_none    = ("resolve_none", "mean"),
).sort_values("n", ascending=False).round(3)
print(pm.to_string())

In [ ]:
# ─── Summary comparison table (ready to paste in a report) ───────────────────
#
# Mirrors the §13 legacy results table from context_for_next_agent.md
# so you can compare directly.

print("=" * 65)
print(" SUMMARY (comparable with legacy R21 results)")
print("=" * 65)
print(f"  {'Metric':<35} {'This run':>10}  {'R21 legacy':>10}")
print("  " + "-" * 57)

rows = [
    ("Hit@window",                   m["hit"].mean(),                          0.220),
    ("gold_in_candidates",           m["gold_in_candidates"].dropna().mean(),   0.235),
    ("resolve_method=none",          m["resolve_none"].mean(),                  0.883),
    ("planner_had_gold_concept",     m["planner_had_gold_concept"].dropna().mean(), 0.137),
    ("Hit | planner_had_gold=True",  had_gold["hit"].mean() if len(had_gold) else float("nan"), 0.338),
]

for label, val, legacy in rows:
    v_str = f"{val:.1%}" if pd.notna(val) else "N/A"
    l_str = f"{legacy:.1%}"
    delta = ""
    if pd.notna(val):
        diff = val - legacy
        delta = f"  (Δ {diff:+.1%})"
    print(f"  {label:<35} {v_str:>10}  {l_str:>10}{delta}")

print("=" * 65)
print("Note: R21 ran on the LEGACY dataset (not comparable structurally).")
print("      Positive Δ = improvement. Negative Δ for resolve_none = improvement.")

In [ ]:
# ─── Top failure cases ────────────────────────────────────────────────────────
# concept_texts that are missed most often — useful for CONCEPT_MAP (§14)

print("=" * 65)
print(" TOP 20 MISSED concept_texts (resolve_none AND hit=False)")
print("=" * 65)

missed = m[(m["resolve_none"] == True) & (m["hit"] == False)]
top_missed = (
    missed.groupby("concept_text")
    .agg(n=("hit", "count"), gold_id=("gold_id", "first"))
    .sort_values("n", ascending=False)
    .head(20)
)
print(top_missed.to_string())

print()
print("─" * 65)
print(" TOP 20 concept_texts where planner missed gold but resolved OK")
print("─" * 65)

planner_miss_but_resolved = m[(m["planner_had_gold_concept"] == False) & (m["resolve_none"] == False) & (m["hit"] == False)]
top_planner = (
    planner_miss_but_resolved.groupby("concept_text")
    .agg(n=("hit", "count"), gold_id=("gold_id", "first"))
    .sort_values("n", ascending=False)
    .head(20)
)
print(top_planner.to_string())

In [ ]:
# ─── Export enriched CSV ─────────────────────────────────────────────────────
# Save a version of the CSV with the computed flags attached.
# Useful for further analysis in other notebooks.

OUT_ENRICHED = Path(EVAL_CSV).with_suffix(".metrics.csv")

export_cols = list(res.columns) + ["gold_id", "split", "hit", "gold_in_candidates",
                                    "planner_had_gold_concept", "resolve_none",
                                    "window_size", "tool_called"]
export_cols = [c for c in export_cols if c in merged.columns]  # safety

merged[export_cols].to_csv(OUT_ENRICHED, index=False)
print(f"Enriched CSV written: {OUT_ENRICHED}")
print(f"  Rows: {len(merged):,}")
print(f"  Cols: {export_cols}")